# Multi-head ModernBERT (plan.md §6 contingency — triggered 2026-07-08)

One shared `answerdotai/ModernBERT-large` encoder, **no** dimension-conditioning text,
`Linear(hidden, 6)` = one output unit per `kb_dim`, masked MSE (each row backprops only
through its own dimension's column). One forward pass yields all 6 scores; structurally
incapable of collapsing across dimensions at the head level.

Triggered by §6 trigger 2: the conditioned model's vh−vl gap on the simulated set was
~uniform across all 6 scored dimensions (targeted dimension never the row max).

**Changes vs. the old scaffold (2026-07-09):** `fp16` → `bf16` (ModernBERT NaNs in fp16 —
hard requirement, A100/L4 only, never T4); batch 8 / grad-accum 2, no gradient
checkpointing (A100 config, matches the conditioned run); dynamic `MAX_LENGTH` from the
p99 of train conversations; early stopping (patience 2); same seeded topic-stratified
split as `ModernBertConditioned.ipynb` (test set stays frozen); gate cells at the end
(simulated set §8a + cross-dimension contrast set §8b, scored with **both** this model
and the saved conditioned checkpoint).

In [23]:
import torch

assert torch.cuda.is_available(), "No GPU. Use an A100 or L4 runtime."
assert torch.cuda.is_bf16_supported(), (
    "GPU has no bf16 support (probably a T4). ModernBERT must NOT be trained in fp16 "
    "(NaN instability) - switch the runtime to A100 or L4."
)
print(torch.cuda.get_device_name(0))

NVIDIA A100-SXM4-80GB


In [24]:
!pip install "transformers>=4.48.0" datasets accelerate scipy scikit-learn --quiet

## Google Drive

Mounted once, up front, so a mount problem surfaces *before* training. Expected layout
under `MyDrive/PhD/AITutorEval/`:

- `data/conversations.jsonl` — the §8a simulated set (required; same file the conditioned run used).
- `data/contrast_conversations.jsonl` — the §8b cross-dimension contrast set (optional at
  train time; the contrast cells at the end skip themselves with a message if it is missing).
  Produce it with `data/run_set_contrast.json` + `evaluations/pedagogy/collect_conversations.py`,
  then upload.
- `models/conditioned_modernbert` — the saved conditioned checkpoint (needed only by the
  final comparison cells).

In [39]:
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/PhD/AITutorEval"
DRIVE_MODELS = f"{DRIVE_BASE}/models"
SIM_PATH = f"{DRIVE_BASE}/data/conversations.jsonl"
CONTRAST_PATH = f"{DRIVE_BASE}/data/contrast_conversations.jsonl"
os.makedirs(DRIVE_MODELS, exist_ok=True)

assert os.path.exists(SIM_PATH), (
    f"Missing {SIM_PATH} - upload runs/convolearn/run_set_07081515/conversations.jsonl "
    "to MyDrive/PhD/AITutorEval/data/ before running."
)
n_sim = sum(1 for _ in open(SIM_PATH, encoding="utf-8"))
print(f"Found simulated set: {SIM_PATH} ({n_sim} conversations)")
if os.path.exists(CONTRAST_PATH):
    n_con = sum(1 for _ in open(CONTRAST_PATH, encoding="utf-8"))
    print(f"Found contrast set:  {CONTRAST_PATH} ({n_con} conversations)")
else:
    print(f"Contrast set not found ({CONTRAST_PATH}) - contrast cells at the end will be skipped.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found simulated set: /content/drive/MyDrive/PhD/AITutorEval/data/conversations.jsonl (60 conversations)
Found contrast set:  /content/drive/MyDrive/PhD/AITutorEval/data/contrast_conversations.jsonl (60 conversations)


In [26]:
from datasets import load_dataset

dataset = load_dataset("masharma/convolearn")
raw = dataset["train"]
print(raw)

Dataset({
    features: ['kb_subdim', 'kb_dim', 'effectiveness_consensus', 'completeness_consensus', 'cleaned_conversation', 'earthscience_topic', 'num_exchanges'],
    num_rows: 2134
})


## Dimension definitions

Copied verbatim from `evaluations/pedagogy/dimensions.md` (the notebook runs on Colab with no
repo access, so the text is embedded). The multi-head model itself never sees this text —
it is needed for (a) the head ordering assert and (b) re-scoring the contrast set with the
*conditioned* checkpoint at the end.

In [27]:
DIMENSIONS = {
    "Cognitive Engagement": (
        "Cognitive Engagement refers to the depth of processing and quality of thinking "
        "strategies students employ during learning (Blumenfeld et al., 2006; Chi & Wylie, 2014). "
        "In dialogic tutoring, cognitive engagement is the most direct behavioral counterpart to "
        "answer-giving: where an answer-giving tutor resolves cognitive challenge by providing "
        "solutions, a dialogically engaging tutor uses that challenge as the site of learning. "
        "Linguistically, it manifests as open-ended questioning, uptake of student ideas, and "
        "scaffolded elaboration rather than declarative explanation."
    ),
    "Formative Assessment": (
        "Formative Assessment refers to the ongoing, interactive monitoring of student "
        "understanding during instruction to regulate learning in real time (Cowie & Bell, 1999; "
        "Black & Wiliam, 2009). Unlike summative evaluation, it is embedded within the dialogue: "
        "tutors attend to student contributions, interpret them against learning goals, and adapt "
        "their next move accordingly. Linguistically, it appears as comprehension checks, probing "
        "follow-up questions, and responses that build on or correct student ideas."
    ),
    "Accountability": (
        "Accountability reflects expectations that discourse aligns with norms of evidence and "
        "reasoning (Michaels et al., 2008). In dialogic tutoring, accountability moves the "
        "conversation beyond mere exchange of opinions toward epistemic responsibility: students "
        "are expected to justify claims, evaluate evidence, and engage with counterarguments. "
        "Linguistically, it manifests as tutor prompts that require students to cite evidence, "
        "explain their reasoning, or defend a position."
    ),
    "Cultural Responsiveness": (
        "Cultural Responsiveness recognizes that effective instruction must engage learners' "
        "diverse cultural backgrounds and build on their prior knowledge and experiences "
        "(Ladson-Billings, 1995; Au et al., 1981). In dialogic tutoring, it requires tutors to "
        "connect concepts to culturally relevant contexts and affirm diverse ways of knowing "
        "rather than assuming a single canonical frame. Linguistically, it manifests as analogies "
        "drawn from students' cultural contexts and invitations to connect content to their "
        "experience."
    ),
    "Metacognition": (
        "Metacognition refers to awareness and regulation of one's own cognitive processes "
        "(Flavell, 1981). In dialogic tutoring, it involves prompting reflection and modeling "
        "through thinking aloud, making expert reasoning visible and encouraging students to "
        "adopt similar habits. Linguistically, it appears as prompts asking students to explain "
        "their reasoning, identify difficulties, or compare current understanding with prior "
        "beliefs."
    ),
    "Power Dynamics": (
        "Power Dynamics captures how agency and participation are distributed in learning "
        "interactions (Gordon & Foucault, 1980). In dialogic tutoring, equitable power dynamics "
        "are essential: tutors who dominate the conversational floor or position themselves as "
        "sole arbiters of knowledge undermine the collaborative construction of understanding. "
        "Linguistically, it manifests in turn-taking patterns, the degree to which student ideas "
        "are taken up and built upon, and whether students are positioned as active contributors."
    ),
}

ALL_DIMS = sorted(DIMENSIONS)  # head order: column i of the output = ALL_DIMS[i]
DIM2IDX = {dim: i for i, dim in enumerate(ALL_DIMS)}
assert set(ALL_DIMS) == set(raw.unique("kb_dim")), (
    "DIMENSIONS keys do not match the dataset's kb_dim values: "
    f"{set(ALL_DIMS) ^ set(raw.unique('kb_dim'))}"
)
print("Head order:", DIM2IDX)

Head order: {'Accountability': 0, 'Cognitive Engagement': 1, 'Cultural Responsiveness': 2, 'Formative Assessment': 3, 'Metacognition': 4, 'Power Dynamics': 5}


In [28]:
from datasets import DatasetDict
import numpy as np

# Same seeded topic-stratified 70/15/15 split as ModernBertConditioned.ipynb - identical
# code, identical seed, so the test set is the same frozen set across both architectures.
topic_to_indices = {}
for i, topic in enumerate(raw["earthscience_topic"]):
    topic_to_indices.setdefault(topic, []).append(i)

train_indices, val_indices, test_indices = [], [], []
rng = np.random.default_rng(seed=42)

for topic, indices in topic_to_indices.items():
    indices = np.array(indices)
    rng.shuffle(indices)
    n = len(indices)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    train_indices.extend(indices[:n_train].tolist())
    val_indices.extend(indices[n_train:n_train + n_val].tolist())
    test_indices.extend(indices[n_train + n_val:].tolist())

dataset_dict = DatasetDict({
    "train": raw.select(train_indices),
    "validation": raw.select(val_indices),
    "test": raw.select(test_indices),
})
print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['kb_subdim', 'kb_dim', 'effectiveness_consensus', 'completeness_consensus', 'cleaned_conversation', 'earthscience_topic', 'num_exchanges'],
        num_rows: 1491
    })
    validation: Dataset({
        features: ['kb_subdim', 'kb_dim', 'effectiveness_consensus', 'completeness_consensus', 'cleaned_conversation', 'earthscience_topic', 'num_exchanges'],
        num_rows: 318
    })
    test: Dataset({
        features: ['kb_subdim', 'kb_dim', 'effectiveness_consensus', 'completeness_consensus', 'cleaned_conversation', 'earthscience_topic', 'num_exchanges'],
        num_rows: 325
    })
})


In [29]:
from transformers import AutoTokenizer

BASE_MODEL = "answerdotai/ModernBERT-large"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# max_length from the 99th percentile of *raw* train conversations (no conditioning text
# here, so inputs are ~150 tokens shorter than the conditioned run's), rounded up to x64.
train_lengths = np.array([
    len(ids) for ids in tokenizer(
        list(dataset_dict["train"]["cleaned_conversation"]), truncation=False
    )["input_ids"]
])
p99 = int(np.percentile(train_lengths, 99))
MAX_LENGTH = min(-(-p99 // 64) * 64, 4096)
print(f"p99 train length: {p99} tokens -> MAX_LENGTH = {MAX_LENGTH}")
print(f"share truncated at {MAX_LENGTH}: {(train_lengths > MAX_LENGTH).mean():.2%}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (13714 > 8192). Running this sequence through the model will result in indexing errors


p99 train length: 1857 tokens -> MAX_LENGTH = 1920
share truncated at 1920: 0.87%


In [30]:
from transformers import DataCollatorWithPadding

def tokenize_multihead(example):
    # Raw conversation only - no dimension-conditioning text.
    encoding = tokenizer(
        example["cleaned_conversation"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False,
    )
    encoding["labels"] = float(example["effectiveness_consensus"])
    encoding["dim_idx"] = DIM2IDX[example["kb_dim"]]
    return encoding

tokenized_mh = dataset_dict.map(
    tokenize_multihead,
    remove_columns=dataset_dict["train"].column_names,
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(tokenized_mh)

Map:   0%|          | 0/325 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'dim_idx'],
        num_rows: 1491
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'dim_idx'],
        num_rows: 318
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'dim_idx'],
        num_rows: 325
    })
})


In [31]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

class MultiHeadModernBertRegressor(nn.Module):
    def __init__(self, base_model_name, num_dims):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name, attn_implementation="sdpa")
        self.heads = nn.Linear(self.encoder.config.hidden_size, num_dims)

    def forward(self, input_ids, attention_mask, dim_idx=None, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        # Attention-masked mean pooling (same pooling as the conditioned run's
        # classifier_pooling="mean").
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        logits = self.heads(pooled)  # (batch, num_dims) - one score per dimension

        loss = None
        if labels is not None and dim_idx is not None:
            # Masked MSE: each row trains only its own dimension's column.
            selected = logits.gather(1, dim_idx.unsqueeze(1)).squeeze(1)
            loss = F.mse_loss(selected, labels.float())

        return {"loss": loss, "logits": logits}

mh_model = MultiHeadModernBertRegressor(BASE_MODEL, num_dims=len(ALL_DIMS))

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
import inspect
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from scipy.stats import pearsonr

def compute_metrics_multihead(eval_pred):
    preds = eval_pred.predictions            # (N, num_dims)
    # label_names order: ["labels", "dim_idx"]. dim_idx rides along as a second
    # "label" because Trainer only forwards input_ids (not arbitrary columns) to metrics.
    labels, dim_idx = eval_pred.label_ids
    labels = np.array(labels)
    dim_idx = np.array(dim_idx, dtype=int)
    selected = preds[np.arange(len(labels)), dim_idx]

    metrics = {
        "overall_mse": float(np.mean((selected - labels) ** 2)),
        "overall_pearson": float(pearsonr(selected, labels)[0]),
    }
    for i, name in enumerate(ALL_DIMS):
        mask = dim_idx == i
        if mask.sum() > 1:
            slug = name.lower().replace(" ", "_")
            metrics[f"{slug}_pearson"] = float(pearsonr(selected[mask], labels[mask])[0])
    return metrics

ta_params = inspect.signature(TrainingArguments.__init__).parameters
compat_kwargs = {}
if "save_safetensors" in ta_params:
    # Removed in transformers v5 (safetensors is the only format there). On v4,
    # keep plain torch.save checkpoints for this custom nn.Module.
    compat_kwargs["save_safetensors"] = False

mh_training_args = TrainingArguments(
    output_dir="./modernbert-convolearn-multihead",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    bf16=True,                       # never fp16 with ModernBERT
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_overall_mse",
    greater_is_better=False,
    logging_steps=50,
    seed=42,
    label_names=["labels", "dim_idx"],
    report_to="none",
    **compat_kwargs,
)

mh_trainer = Trainer(
    model=mh_model,
    args=mh_training_args,
    train_dataset=tokenized_mh["train"],
    eval_dataset=tokenized_mh["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics_multihead,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

mh_trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Overall Mse,Overall Pearson,Accountability Pearson,Cognitive Engagement Pearson,Cultural Responsiveness Pearson,Formative Assessment Pearson,Metacognition Pearson,Power Dynamics Pearson
1,4.427786,0.794422,0.794422,0.581299,0.284388,0.634708,0.043241,0.591946,0.673309,0.605267
2,0.785101,0.713296,0.713296,0.625915,0.424113,0.744330,0.043651,0.670723,0.710172,0.695888
3,0.555635,0.651573,0.651573,0.656068,0.400394,0.753264,0.073969,0.702078,0.715839,0.725146
4,0.425480,0.762668,0.762668,0.641573,0.197590,0.750358,0.101486,0.722801,0.694472,0.683641
5,0.278840,0.667457,0.667457,0.656537,0.344335,0.763235,0.097693,0.698827,0.707690,0.691775


TrainOutput(global_step=470, training_loss=0.9564511278842358, metrics={'train_runtime': 342.1752, 'train_samples_per_second': 21.787, 'train_steps_per_second': 1.374, 'total_flos': 0.0, 'train_loss': 0.9564511278842358, 'epoch': 5.0})

## Test-set metrics (frozen split, comparable to the conditioned run)

Conditioned ModernBERT reference (2026-07-08): overall r=0.702 / RMSE=0.767 / MAE=0.578;
per-dim r: Acc .749, CE .688, CR .178, FA .816, Meta .702, PD .711.

In [33]:
from scipy.stats import spearmanr

pred_out = mh_trainer.predict(tokenized_mh["test"])
preds_all = pred_out.predictions                      # (N, 6)
labels_te = np.array(pred_out.label_ids[0])
dim_idx_te = np.array(pred_out.label_ids[1], dtype=int)
selected = preds_all[np.arange(len(labels_te)), dim_idx_te]

def _row(name, p, l):
    return (f"{name:<26} n={len(l):<5} r={pearsonr(p, l)[0]:.3f}  "
            f"rho={spearmanr(p, l)[0]:.3f}  "
            f"RMSE={np.sqrt(np.mean((p - l) ** 2)):.3f}  MAE={np.mean(np.abs(p - l)):.3f}")

print(_row("OVERALL", selected, labels_te))
print("-" * 80)
for i, name in enumerate(ALL_DIMS):
    mask = dim_idx_te == i
    if mask.sum() > 1:
        print(_row(name, selected[mask], labels_te[mask]))

OVERALL                    n=325   r=0.656  rho=0.665  RMSE=0.829  MAE=0.655
--------------------------------------------------------------------------------
Accountability             n=45    r=0.749  rho=0.753  RMSE=0.771  MAE=0.652
Cognitive Engagement       n=83    r=0.573  rho=0.569  RMSE=0.823  MAE=0.678
Cultural Responsiveness    n=24    r=0.185  rho=0.157  RMSE=1.047  MAE=0.798
Formative Assessment       n=36    r=0.764  rho=0.750  RMSE=0.742  MAE=0.579
Metacognition              n=87    r=0.681  rho=0.651  RMSE=0.735  MAE=0.587
Power Dynamics             n=50    r=0.640  rho=0.680  RMSE=0.970  MAE=0.723


In [34]:
import json, shutil

mh_model_path = "./models/multihead_modernbert"
os.makedirs(mh_model_path, exist_ok=True)

# Custom nn.Module, not a Trainer-native model class - save state_dict + tokenizer manually.
torch.save(mh_trainer.model.state_dict(), f"{mh_model_path}/state_dict.pt")
tokenizer.save_pretrained(mh_model_path)
with open(f"{mh_model_path}/run_config.json", "w") as f:
    json.dump({"base_model": BASE_MODEL, "max_length": MAX_LENGTH, "dims": ALL_DIMS}, f, indent=2)

shutil.copytree(mh_model_path, f"{DRIVE_MODELS}/multihead_modernbert", dirs_exist_ok=True)
print(f"Saved and copied to {DRIVE_MODELS}/multihead_modernbert")

Saved and copied to /content/drive/MyDrive/PhD/AITutorEval/models/multihead_modernbert


## Gate: simulated set (§8a) — discrimination check

Same 3×6 vh−vl gap matrix and per-dimension AUC as the conditioned notebook, for direct
comparison. The conditioned model's result (2026-07-08): AUC on targeted dim .96–1.0
(validity passed), but the gap was ~uniform across all 6 dims and the targeted dim was
never the row max (discrimination failed).

In [35]:
@torch.no_grad()
def score_all_mh(texts, batch_size=16):
    """Score conversations on all 6 dimensions in one pass. Returns (N, 6), columns in ALL_DIMS order."""
    model = mh_trainer.model.eval()
    device = next(model.parameters()).device
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[i:i + batch_size], max_length=MAX_LENGTH,
            truncation=True, padding=True, return_tensors="pt",
        ).to(device)
        out.append(model(**enc)["logits"].float().cpu().numpy())
    return np.concatenate(out)

LABEL2DIM = {"ac": "Accountability", "ce": "Cognitive Engagement", "fa": "Formative Assessment"}

sim = [json.loads(line) for line in open(SIM_PATH, encoding="utf-8")]
sim_scores = score_all_mh([r["text"] for r in sim])
print(f"Scored {len(sim)} simulated conversations on all {len(ALL_DIMS)} dimensions.")

Scored 60 simulated conversations on all 6 dimensions.


In [36]:
from sklearn.metrics import roc_auc_score

short = {d: "".join(w[0] for w in d.split()) for d in ALL_DIMS}  # e.g. "CE", "FA"
header = f"{'':<28}" + "".join(f"{short[d]:>8}" for d in ALL_DIMS)
print("vh - vl mean-score gap per scored dimension (* = targeted dim; should be the row max):")
print(header)

for code_, dim in LABEL2DIM.items():
    vh = [i for i, r in enumerate(sim) if r["label"] == f"{code_}-vh"]
    vl = [i for i, r in enumerate(sim) if r["label"] == f"{code_}-vl"]
    gaps = sim_scores[vh].mean(axis=0) - sim_scores[vl].mean(axis=0)
    cells = "".join(
        f"{g:>+7.2f}{'*' if ALL_DIMS[j] == dim else ' '}" for j, g in enumerate(gaps)
    )
    j = ALL_DIMS.index(dim)
    auc = roc_auc_score(
        [1] * len(vh) + [0] * len(vl),
        np.concatenate([sim_scores[vh, j], sim_scores[vl, j]]),
    )
    flag = "OK " if gaps[j] == gaps.max() else "NOT-MAX"
    print(f"{code_} ({dim[:20]:<20}) {cells}   AUC(target)={auc:.3f} [{flag}]")

vh - vl mean-score gap per scored dimension (* = targeted dim; should be the row max):
                                   A      CE      CR      FA       M      PD
ac (Accountability      )   +0.19*  +0.13   +0.15   +0.15   +0.03   +0.18    AUC(target)=0.770 [OK ]
ce (Cognitive Engagement)   +0.11   +0.13*  +0.15   +0.08   +0.14   +0.23    AUC(target)=0.790 [NOT-MAX]
fa (Formative Assessment)   +0.06   +0.20   +0.19   +0.04*  +0.36   +0.38    AUC(target)=0.580 [NOT-MAX]


## Contrast set (§8b) — the decisive diagnostic

Conversations engineered **Very High on dimension A and Very Low on dimension B**
simultaneously (labels like `acvh-cevl`). Pass criterion per conversation: score on A >
score on B, with the margin flipping in the mirrored cell.

- Multi-head passes & conditioned fails → the step-3 failure was architectural; multi-head becomes primary.
- Conditioned passes → the step-3 "failure" was an eval-set confound (§8a vh/vl items co-vary on all dims); conditioned stands.
- Both fail → dimensions are likely not separable from CONVOLEARN's single-dimension labels; stop tuning architectures.

In [40]:
CODE2DIM = {"ac": "Accountability", "ce": "Cognitive Engagement", "fa": "Formative Assessment"}

def parse_contrast_label(label):
    """'acvh-cevl' -> ('Accountability', 'Cognitive Engagement')  (high-dim, low-dim)."""
    hi_tok, lo_tok = label.split("-")
    assert hi_tok.endswith("vh") and lo_tok.endswith("vl"), f"bad contrast label: {label}"
    return CODE2DIM[hi_tok[:-2]], CODE2DIM[lo_tok[:-2]]

def contrast_report(scores_by_dim, labels, model_name):
    """scores_by_dim: {dim_name: (N,) array}. Prints one row per ordered (high, low) pair."""
    from collections import defaultdict
    cells = defaultdict(list)
    for i, lab in enumerate(labels):
        hi, lo = parse_contrast_label(lab)
        cells[(hi, lo)].append((scores_by_dim[hi][i], scores_by_dim[lo][i]))

    print(f"=== {model_name} ===")
    print(f"{'cell (VH-dim / VL-dim)':<46} {'n':>3} {'mean s(VH)':>10} {'mean s(VL)':>10} "
          f"{'margin':>8} {'sign-acc':>9}")
    accs = []
    for (hi, lo) in sorted(cells):
        arr = np.array(cells[(hi, lo)])
        margin = arr[:, 0] - arr[:, 1]
        acc = float((margin > 0).mean())
        accs.append(acc)
        print(f"{hi[:22]:<23}/{lo[:21]:<22} {len(arr):>3} {arr[:, 0].mean():>10.2f} "
              f"{arr[:, 1].mean():>10.2f} {margin.mean():>+8.2f} {acc:>9.0%}")
    print(f"{'mean sign-accuracy':<46} {'':>3} {'':>10} {'':>10} {'':>8} {np.mean(accs):>9.0%}"
          "   (chance = 50%)")
    return float(np.mean(accs))

if os.path.exists(CONTRAST_PATH):
    contrast = [json.loads(line) for line in open(CONTRAST_PATH, encoding="utf-8")]
    contrast_texts = [r["text"] for r in contrast]
    contrast_labels = [r["label"] for r in contrast]
    mh_scores = score_all_mh(contrast_texts)
    mh_by_dim = {d: mh_scores[:, j] for j, d in enumerate(ALL_DIMS)}
    mh_acc = contrast_report(mh_by_dim, contrast_labels, "Multi-head ModernBERT")
else:
    print(f"SKIPPED - no contrast set at {CONTRAST_PATH}.")
    print("Generate it: uv run python -m tutoring_check --run-set data/run_set_contrast.json "
          "--out runs/contrast/run_set_<ts>")
    print("then: uv run python evaluations/pedagogy/collect_conversations.py runs/contrast/run_set_<ts>")
    print("and upload conversations.jsonl to Drive as data/contrast_conversations.jsonl")

=== Multi-head ModernBERT ===
cell (VH-dim / VL-dim)                           n mean s(VH) mean s(VL)   margin  sign-acc
Accountability         /Cognitive Engagement    10       4.70       4.66    +0.05       70%
Accountability         /Formative Assessment    10       4.62       4.54    +0.09       80%
Cognitive Engagement   /Accountability          10       4.72       4.68    +0.04       50%
Cognitive Engagement   /Formative Assessment    10       4.73       4.57    +0.16       90%
Formative Assessment   /Accountability          10       4.47       4.54    -0.08        0%
Formative Assessment   /Cognitive Engagement    10       4.51       4.72    -0.21        0%
mean sign-accuracy                                                                      48%   (chance = 50%)


### Conditioned checkpoint on the same contrast set

Loads `models/conditioned_modernbert` from Drive (saved by `ModernBertConditioned.ipynb`)
and scores the identical conversations, so the two architectures are compared on exactly
the same data in the same session.

In [41]:
COND_PATH = f"{DRIVE_MODELS}/conditioned_modernbert"

def format_input(conversation_text, dim_name):
    return (
        f"Dimension: {dim_name}\n"
        f"Definition: {DIMENSIONS[dim_name]}\n\n"
        f"Conversation:\n{conversation_text}"
    )

if os.path.exists(CONTRAST_PATH) and os.path.exists(COND_PATH):
    from transformers import AutoModelForSequenceClassification

    cond_cfg = json.load(open(f"{COND_PATH}/run_config.json"))
    cond_tok = AutoTokenizer.from_pretrained(COND_PATH)
    cond_model = AutoModelForSequenceClassification.from_pretrained(COND_PATH).cuda().eval()

    @torch.no_grad()
    def score_conditioned(texts, dim_name, batch_size=16):
        out = []
        for i in range(0, len(texts), batch_size):
            enc = cond_tok(
                [format_input(t, dim_name) for t in texts[i:i + batch_size]],
                max_length=cond_cfg["max_length"], truncation=True,
                padding=True, return_tensors="pt",
            ).to(cond_model.device)
            out.append(cond_model(**enc).logits.squeeze(-1).float().cpu().numpy())
        return np.concatenate(out)

    cond_by_dim = {d: score_conditioned(contrast_texts, d) for d in CODE2DIM.values()}
    cond_acc = contrast_report(cond_by_dim, contrast_labels, "Conditioned ModernBERT")
elif not os.path.exists(COND_PATH):
    print(f"SKIPPED - no conditioned checkpoint at {COND_PATH}.")
else:
    print("SKIPPED - contrast set missing (see previous cell).")

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

=== Conditioned ModernBERT ===
cell (VH-dim / VL-dim)                           n mean s(VH) mean s(VL)   margin  sign-acc
Accountability         /Cognitive Engagement    10       4.90       4.96    -0.06        0%
Accountability         /Formative Assessment    10       5.08       5.07    +0.01       80%
Cognitive Engagement   /Accountability          10       5.12       5.02    +0.10      100%
Cognitive Engagement   /Formative Assessment    10       5.06       5.00    +0.06      100%
Formative Assessment   /Accountability          10       4.94       4.92    +0.02       90%
Formative Assessment   /Cognitive Engagement    10       4.91       5.00    -0.08        0%
mean sign-accuracy                                                                      62%   (chance = 50%)


## Interpreting the results

**Test metrics:** the multi-head only earns consideration if its per-dimension r on
Accountability / Cognitive Engagement / Formative Assessment is in the same range as the
conditioned run (.69–.82). A large drop means the conditioning text was doing real work.

**Simulated-set gap matrix:** targeted dimension should now be the row max (it never was
for the conditioned model).

**Contrast set (decides plan step 4b):** compare mean sign-accuracy between the two
models. ≥80% = clear pass; ~50% = no construct-specificity. See the decision table above;
record the outcome in `plan.md` §8 rows 4/4b either way.

### Reloading without retraining

```python
mh_model = MultiHeadModernBertRegressor("answerdotai/ModernBERT-large", num_dims=len(ALL_DIMS))
mh_model.load_state_dict(torch.load(f"{DRIVE_MODELS}/multihead_modernbert/state_dict.pt"))
mh_model.eval()
```

(The class cell must be run first — it's a custom `nn.Module`, not reconstructable from a config.)